# Business Impact & Revenue Simulation Notebook

This notebook walks through:
- Full NBA pipeline execution
- Archetype-level business impact analysis
- Sensitivity analysis (ROI under different assumptions)
- Cohort-level revenue waterfall
- Executive-ready summary tables

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

from src.data.generate_data import generate_dataset
from src.features.feature_engineering import engineer_features, MODEL_FEATURES
from src.nba_engine.archetype_classifier import classify_archetypes, ARCHETYPES
from src.nba_engine.nba_engine import run_nba_pipeline
from src.clv.clv_calculator import add_clv_columns, get_clv_summary
from src.business_impact.impact_engine import (
    simulate_business_impact, print_impact_report, ImpactAssumptions
)

import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

ARCHETYPE_COLORS = {'A': '#6c757d', 'B': '#dc3545', 'C': '#fd7e14', 'D': '#198754'}
ARCHETYPE_LABELS = {
    'A': 'The Ghost', 'B': 'Frustrated Pro',
    'C': 'Price Sensitive', 'D': 'Outgrown User'
}
print('Setup complete ✓')

## 1. Generate Data & Score Customers

In [ ]:
df_raw = generate_dataset(n=8000)
df_eng = engineer_features(df_raw)

X = df_eng[MODEL_FEATURES].fillna(0).values
y = df_eng['churned'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_s = scaler.transform(X)

model = lgb.LGBMClassifier(n_estimators=200, class_weight='balanced', random_state=42, verbose=-1)
model.fit(X_train_s, y_train)

df_eng['churn_probability'] = model.predict_proba(X_s)[:, 1]
df_eng['primary_driver'] = 'engagement_decay_score'

df_hr = df_eng[df_eng['churn_probability'] >= 0.5].copy().reset_index(drop=True)
print(f'Total customers: {len(df_eng):,}')
print(f'High-risk (≥50%): {len(df_hr):,} ({len(df_hr)/len(df_eng):.1%})')

## 2. Run Full NBA Pipeline

In [ ]:
df_hr = classify_archetypes(df_hr)
df_hr = add_clv_columns(df_hr)
df_nba = run_nba_pipeline(df_hr)

report, df_impact = simulate_business_impact(df_nba)
print_impact_report(report)

## 3. Revenue Impact by Archetype

In [ ]:
if not report.by_archetype.empty:
    arch = report.by_archetype.copy()
    arch['color'] = arch['churn_archetype'].map(ARCHETYPE_COLORS)
    arch['label'] = arch['churn_archetype'].map(ARCHETYPE_LABELS)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Revenue saved
    bars = axes[0].bar(arch['label'], arch['revenue_saved'], color=arch['color'], alpha=0.85)
    axes[0].set_title('Expected Revenue Saved ($)', fontweight='bold')
    axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    axes[0].tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, arch['revenue_saved']):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                     f'${val:,.0f}', ha='center', va='bottom', fontsize=9)

    # Customer count
    axes[1].bar(arch['label'], arch['users'], color=arch['color'], alpha=0.85)
    axes[1].set_title('Customers at Risk by Archetype', fontweight='bold')
    axes[1].tick_params(axis='x', rotation=20)

    # Net profit
    colors_profit = ['#198754' if v >= 0 else '#dc3545' for v in arch['net_profit']]
    axes[2].bar(arch['label'], arch['net_profit'], color=colors_profit, alpha=0.85)
    axes[2].set_title('Net Incremental Profit ($)', fontweight='bold')
    axes[2].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    axes[2].tick_params(axis='x', rotation=20)
    axes[2].axhline(0, color='black', linewidth=0.8)

    plt.suptitle('Business Impact Analysis by Churn Archetype', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../reports/nba/impact_by_archetype.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. CLV Tier Analysis

In [ ]:
clv_summary = get_clv_summary(df_hr)
print('CLV Summary by Tier:')
print(clv_summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

tier_order = ['Enterprise', 'High', 'Medium', 'Low']
tier_colors = ['#1a1a2e', '#e94560', '#457b9d', '#a8b2d8']
tier_data = clv_summary.set_index('clv_tier').reindex(tier_order, fill_value=0)

axes[0].bar(tier_data.index, tier_data['count'], color=tier_colors, alpha=0.85)
axes[0].set_title('Customers by CLV Tier', fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].bar(tier_data.index, tier_data['avg_clv'], color=tier_colors, alpha=0.85)
axes[1].set_title('Average CLV by Tier ($)', fontweight='bold')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

## 5. ROI Sensitivity Analysis

In [ ]:
# Sensitivity: ROI as conversion rate and reach rate vary
conv_rates = np.arange(0.05, 0.50, 0.05)
reach_rates = [0.60, 0.75, 0.85, 1.00]

fig, ax = plt.subplots(figsize=(11, 6))
line_styles = ['-', '--', '-.', ':']

for reach, ls in zip(reach_rates, line_styles):
    rois = []
    for conv in conv_rates:
        assumptions = ImpactAssumptions(
            campaign_reach_rate=reach,
            months_retained=12,
            incremental_margin=0.70,
            overhead_multiplier=1.20,
        )
        df_sim = df_nba.copy()
        df_sim['expected_conversion_rate'] = conv
        r, _ = simulate_business_impact(df_sim, assumptions)
        rois.append(r.roi_pct)
    ax.plot(conv_rates * 100, rois, linestyle=ls, linewidth=2, marker='o', markersize=5,
            label=f'Reach={reach:.0%}')

ax.set_xlabel('Conversion Rate (%)', fontsize=12)
ax.set_ylabel('ROI (%)', fontsize=12)
ax.set_title('ROI Sensitivity — Conversion Rate × Campaign Reach Rate', fontsize=13, fontweight='bold')
ax.legend(title='Reach Rate', framealpha=0.9)
ax.axhline(0, color='red', linewidth=1, linestyle='--', alpha=0.5, label='Break-even')
plt.tight_layout()
plt.savefig('../reports/nba/roi_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Revenue Waterfall Chart

In [ ]:
# Revenue waterfall: at-risk → contacted → saved → profit
total_mrr_atrisk = df_nba['monthly_revenue'].sum() * 12
contacted = total_mrr_atrisk * 0.85
converted_rev = report.expected_revenue_saved
campaign_cost = report.total_campaign_cost
net_profit = report.net_incremental_profit

labels = ['Annual MRR\nat Risk', 'Reached\nby Campaign', 'Gross Revenue\nSaved', 'Campaign\nCost', 'Net\nProfit']
values = [total_mrr_atrisk, -total_mrr_atrisk + contacted, converted_rev - contacted, -campaign_cost, net_profit - converted_rev + campaign_cost]

# Simpler waterfall
display_vals = [total_mrr_atrisk, contacted, converted_rev, -campaign_cost, net_profit]
display_labels = ['MRR at Risk (12mo)', 'Reached', 'Revenue Saved', 'Campaign Cost', 'Net Profit']
bar_colors = ['#457b9d', '#a8d8ea', '#2ecc71', '#e74c3c', '#27ae60']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(display_labels, display_vals, color=bar_colors, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, display_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + abs(max(display_vals)) * 0.01,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Revenue Impact Waterfall — Retention Campaign', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_ylabel('USD')
plt.tight_layout()
plt.savefig('../reports/nba/revenue_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Executive Summary Table

In [ ]:
exec_summary = pd.DataFrame([
    {'Metric': 'Total Customers Analysed', 'Value': f'{len(df_eng):,}'},
    {'Metric': 'High-Risk Customers (≥50%)', 'Value': f'{len(df_hr):,} ({len(df_hr)/len(df_eng):.1%})'},
    {'Metric': 'Annual MRR at Risk', 'Value': f'${report.total_at_risk_mrr*12:,.0f}'},
    {'Metric': 'Expected Customers Retained', 'Value': f'{report.expected_users_retained:,.0f}'},
    {'Metric': 'Expected Revenue Saved (12mo)', 'Value': f'${report.expected_revenue_saved:,.0f}'},
    {'Metric': 'Total Campaign Cost', 'Value': f'${report.total_campaign_cost:,.0f}'},
    {'Metric': 'Net Incremental Profit', 'Value': f'${report.net_incremental_profit:,.0f}'},
    {'Metric': 'Campaign ROI', 'Value': f'{report.roi_pct:.0f}%'},
    {'Metric': 'Payback Period', 'Value': f'{report.payback_months:.1f} months'},
    {'Metric': 'Champion Model', 'Value': 'LightGBM (ROC-AUC ≥ 0.68)'},
    {'Metric': 'Most Common Archetype', 'Value': df_nba['churn_archetype'].value_counts().index[0]},
    {'Metric': 'Highest-ROI Action', 'Value': df_impact.groupby('action')['revenue_saved'].sum().idxmax() if 'revenue_saved' in df_impact.columns else 'N/A'},
])

print('='*55)
print('  NBA CHURN RETENTION ENGINE — EXECUTIVE SUMMARY')
print('='*55)
print(exec_summary.to_string(index=False))
print('='*55)

exec_summary.to_csv('../reports/nba/executive_summary.csv', index=False)
print('Saved to reports/nba/executive_summary.csv')